In [ ]:
!pip install pdfplumber

In [4]:
import pdfplumber
import requests
import io
import pandas as pd

# Fetch and open the PDF
response = requests.get("https://www.lta.gov.sg/content/dam/ltagov/who_we_are/statistics_and_publications/statistics/pdf/PT_Ridership_Monthly_2019-2025.pdf")
with pdfplumber.open(io.BytesIO(response.content)) as pdf:
    # Extract table from the first page
    first_page = pdf.pages[0]
    table_data = first_page.extract_table()

    # Convert to DataFrame (using the first row as headers)
    df = pd.DataFrame(table_data[1:], columns=table_data[0])


In [5]:
df

,Month,Public Bus,MRT,LRT
0,Jan 2019,"4,250","3,462",218
1,Feb 2019,"4,010","3,248",206
2,Mar 2019,"4,111","3,383",209
3,Apr 2019,"4,189","3,400",213
4,May 2019,"4,199","3,359",215
5,Jun 2019,"3,840","3,246",194
6,Jul 2019,"4,300","3,551",218
7,Aug 2019,"4,163","3,419",211
8,Sep 2019,"4,117","3,444",208
9,Oct 2019,"4,170","3,431",211


In [6]:
# 1. Fetch the online PDF
url = "https://www.lta.gov.sg/content/dam/ltagov/who_we_are/statistics_and_publications/statistics/pdf/PT_Ridership_Monthly_2019-2025.pdf"
response = requests.get(url)
pdf_content = io.BytesIO(response.content)

# 2. Initialize a list to hold all dataframes
all_tables = []

# 3. Open the PDF and loop through every page
with pdfplumber.open(pdf_content) as pdf:
    for i, page in enumerate(pdf.pages):
        # Extract all tables on the current page
        tables = page.extract_tables()

        for table in tables:
            # Convert the list of lists into a DataFrame
            # Uses the first row of the table as the header
            df = pd.DataFrame(table[1:], columns=table[0])

            # Optional: Add a column to track which page the data came from
            df['source_page'] = i + 1
            all_tables.append(df)

# 4. Concatenate all individual tables into one master DataFrame
if all_tables:
    final_df = pd.concat(all_tables, ignore_index=True)
    print("Extraction complete. Combined shape:", final_df.shape)
else:
    print("No tables were found in the PDF.")

Extraction complete. Combined shape: (84, 5)


In [7]:
final_df

,Month,Public Bus,MRT,LRT,source_page
0,Jan 2019,"4,250","3,462",218,1
1,Feb 2019,"4,010","3,248",206,1
2,Mar 2019,"4,111","3,383",209,1
3,Apr 2019,"4,189","3,400",213,1
4,May 2019,"4,199","3,359",215,1
...,...,...,...,...,...
79,Aug 2025,"4,001","3,668",215,7
80,Sep 2025,"3,899","3,530",210,7
81,Oct 2025,"3,909","3,581",211,7
82,Nov 2025,"3,818","3,495",204,7


In [8]:
final_df.to_csv('PT_Ridership_Monthly.csv', index=False)

In [9]:
list(final_df)

['Month', 'Public Bus', 'MRT', 'LRT', 'source_page']